## Imports and Setup
Loading required libraries for API calls, validation, and model handling.

In [15]:
import os
import requests
import json
import re

from dotenv import load_dotenv
from jsonschema import validate, ValidationError

import joblib
import pandas as pd
import numpy as np

## Load API Key
Loading the LLM API key securely from environment variables.

In [16]:
load_dotenv()

api_key = os.getenv("GROQ_API_KEY")

print("API key loaded:", api_key is not None)

API key loaded: True


## LLM API Function
Reusable function to send prompts and receive structured responses.

In [17]:
def call_llm(system_prompt, user_prompt, temperature=0.0, max_tokens=512):

    url = "https://api.groq.com/openai/v1/chat/completions"


    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }


    payload = {

        "model": "llama-3.1-8b-instant",

        "messages": [
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ],

        "temperature": temperature,

        "max_tokens": max_tokens
    }


    response = requests.post(
        url,
        headers=headers,
        json=payload
    )


    if response.status_code != 200:
        print("Error:", response.status_code)
        print(response.text)
        return None


    return response.json()["choices"][0]["message"]["content"]

## Test LLM Connection
Checking whether the API returns a valid response.

In [18]:
def call_llm(system_prompt, user_prompt, temperature=0.0, max_tokens=512):

    url = "https://api.groq.com/openai/v1/chat/completions"


    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }


    payload = {

        "model": "llama-3.1-8b-instant",

        "messages": [
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ],

        "temperature": temperature,

        "max_tokens": max_tokens
    }


    response = requests.post(
        url,
        headers=headers,
        json=payload
    )


    if response.status_code != 200:
        print("Error:", response.status_code)
        print(response.text)
        return None


    return response.json()["choices"][0]["message"]["content"]

In [19]:
test_response = call_llm(
    "You are a helpful assistant.",
    "Reply with only the word: hello"
)

print(test_response)

hello


## PII Guardrail
Blocking inputs that contain personally identifiable information before sending them to the LLM.

In [20]:
def has_pii(text):

    email_pattern = r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+'

    phone_pattern = r'\b\d{10}\b|\b\d{3}[-.\s]\d{3}[-.\s]\d{4}\b'

    return bool(
        re.search(email_pattern, text) or
        re.search(phone_pattern, text)
    )

## Test PII Detection
Testing blocked and allowed inputs.

In [21]:
test_inputs = [
    "Contact me at example@gmail.com",
    "Explain this house prediction"
]


for text in test_inputs:

    if has_pii(text):
        print("Input blocked: PII detected.")
    else:
        print("Input allowed:", text)

Input blocked: PII detected.
Input allowed: Explain this house prediction


## Load Trained Model
Loading the best Random Forest pipeline from Part 3.

In [22]:
model_path = (
    "../Part-3-Ensemble-Models-and-Pipelines/"
    "best_model.pkl"
)


model = joblib.load(model_path)


print("Model loaded successfully")

Model loaded successfully


## Prepare Feature Inputs
Loading sample records to generate model predictions.

In [23]:
data_path = (
    "../Part-1-Data-Acquisition-and-EDA/"
    "cleaned_data.csv"
)


df = pd.read_csv(data_path)


df.head()

,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,...,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
0,1,526301100,20,RL,141.0,31770,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,NaN,0,5,2010,WD,Normal,215000
1,2,526350040,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,...,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal,105000
2,3,526351010,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal,172000
3,4,526353030,20,RL,93.0,11160,Pave,NaN,Reg,Lvl,...,0,NaN,NaN,NaN,0,4,2010,WD,Normal,244000
4,5,527105010,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,...,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal,189900


## Create Three Test Records
Selecting three feature vectors for LLM explanation.

In [24]:
# Remove target column

X_data = df.drop(
    "SalePrice",
    axis=1
)


# Select three examples

test_records = [
    X_data.iloc[0].to_dict(),
    X_data.iloc[1].to_dict(),
    X_data.iloc[2].to_dict()
]


len(test_records)

3

## Model Prediction
Generating class predictions and confidence scores.

In [25]:
def predict_record(record):

    input_df = pd.DataFrame([record])


    prediction = model.predict(
        input_df
    )[0]


    probability = model.predict_proba(
        input_df
    )[0][1]


    return prediction, probability

In [26]:
for i, record in enumerate(test_records):

    pred, prob = predict_record(record)

    print("Record:", i+1)
    print("Prediction:", pred)
    print("Probability:", prob)
    print("----------------")

ValueError: The feature names should match those that were passed during fit.
Feature names unseen at fit time:
- Alley
- Bldg Type
- Bsmt Cond
- Bsmt Exposure
- Bsmt Qual
- ...
Feature names seen at fit time, yet now missing:
- Alley_Pave
- Bldg Type_2fmCon
- Bldg Type_Duplex
- Bldg Type_Twnhs
- Bldg Type_TwnhsE
- ...


## Prepare Encoded Feature Inputs
Applying the same preprocessing used during model training.

In [27]:
import pandas as pd

df = pd.read_csv(
    "../Part-1-Data-Acquisition-and-EDA/cleaned_data.csv"
)

In [28]:
X = df.drop(
    "SalePrice",
    axis=1
)

In [29]:
X_encoded = pd.get_dummies(
    X,
    drop_first=True
)

In [30]:
model_features = model.named_steps["standardscaler"].feature_names_in_

X_encoded = X_encoded.reindex(
    columns=model_features,
    fill_value=0
)

KeyError: 'standardscaler'

In [31]:
test_records = [
    X_encoded.iloc[0].to_dict(),
    X_encoded.iloc[1].to_dict(),
    X_encoded.iloc[2].to_dict()
]

In [32]:
def predict_record(record):

    input_df = pd.DataFrame([record])


    prediction = model.predict(
        input_df
    )[0]


    probability = model.predict_proba(
        input_df
    )[0][1]


    return prediction, probability

In [33]:
for i, record in enumerate(test_records):

    pred, prob = predict_record(record)

    print("Record:", i+1)
    print("Prediction:", pred)
    print("Probability:", prob)
    print("----------------")

Record: 1
Prediction: 1
Probability: 0.88
----------------
Record: 2
Prediction: 0
Probability: 0.04
----------------
Record: 3
Prediction: 1
Probability: 0.735
----------------


## Inspect Saved Model Pipeline
Checking the preprocessing steps stored inside the trained model.

In [34]:
model.named_steps

{'imputer': SimpleImputer(strategy='median'),
 'scaler': StandardScaler(),
 'randomforestclassifier': RandomForestClassifier(n_estimators=200, random_state=42)}

## JSON Output Schema
Defining the required structure for LLM explanations.

In [42]:
explanation_schema = {

    "type": "object",

    "properties": {

        "prediction_label": {
            "type": "string"
        },

        "confidence_level": {
            "type": "string"
        },

        "top_reason": {
            "type": "string"
        },

        "second_reason": {
            "type": "string"
        },

        "next_step": {
            "type": "string"
        }
    },

    "required": [
        "prediction_label",
        "confidence_level",
        "top_reason",
        "second_reason",
        "next_step"
    ]
}

## LLM Explanation Pipeline
Generating structured explanations for model predictions.

In [43]:
def generate_explanation(record, prediction, probability):


    # Use only selected features to reduce prompt size
    important_features = {
        k: v 
        for k, v in list(record.items())[:10]
    }


    system_prompt = """
You are a machine learning explanation assistant.

Your task is to explain house price classification predictions.

Return ONLY valid JSON.
Do not use markdown.
Do not add extra text.

The JSON must contain:
prediction_label,
confidence_level,
top_reason,
second_reason,
next_step
"""


    user_prompt = f"""

Important house features:
{important_features}


Model prediction:
{prediction}


Prediction probability:
{probability}


Rules:

prediction_label:
Return only "High Price" or "Low Price"

confidence_level:
Return only "low", "medium", or "high"

Explain the prediction briefly.

Return JSON only.
"""


    # PII check

    if has_pii(user_prompt):

        print("Input blocked: PII detected.")

        return None



    response = call_llm(
        system_prompt,
        user_prompt,
        temperature=0
    )


    if response is None:
        return {
            "prediction_label": None,
            "confidence_level": None,
            "top_reason": None,
            "second_reason": None,
            "next_step": None
        }


    try:

        result = json.loads(
            response.strip()
        )


        validate(
            instance=result,
            schema=explanation_schema
        )


        return result


    except (json.JSONDecodeError, ValidationError) as e:

        print("Validation failed:", e)


        return {
            "prediction_label": None,
            "confidence_level": None,
            "top_reason": None,
            "second_reason": None,
            "next_step": None
        }

## Final LLM Explanation Results
Running predictions and generating validated explanations.

In [44]:
results = []


for i, record in enumerate(test_records):

    prediction, probability = predict_record(record)


    explanation = generate_explanation(
        record,
        prediction,
        probability
    )


    results.append({

        "Input": i+1,

        "Prediction": prediction,

        "Probability": probability,

        "Explanation": explanation

    })


results

[{'Input': 1,
  'Prediction': np.int64(1),
  'Probability': np.float64(0.88),
  'Explanation': {'prediction_label': 'High Price',
   'confidence_level': 'high',
   'top_reason': 'High Lot Frontage (141.0) and Overall Qual (6) indicate a high-end property',
   'second_reason': 'Year Built (1960) and Year Remod/Add (1960) suggest a well-maintained property',
   'next_step': 'Verify property details and inspect the property for any potential issues'}},
 {'Input': 2,
  'Prediction': np.int64(0),
  'Probability': np.float64(0.04),
  'Explanation': {'prediction_label': 'Low Price',
   'confidence_level': 'high',
   'top_reason': 'The house was built in 1961, which is relatively old, and the overall condition is 6, indicating some wear and tear.',
   'second_reason': 'The lot area is 11622, which is relatively small compared to other houses in the area.',
   'next_step': 'Consider inspecting the house more closely to determine the extent of any needed repairs before making an offer.'}},
 {'In

## Temperature Comparison
Comparing deterministic and creative LLM responses at different temperature values.

In [45]:
temperature_results = []


for i, record in enumerate(test_records):

    prediction, probability = predict_record(record)


    prompt_record = {
        k: v 
        for k, v in list(record.items())[:10]
    }


    system_prompt = """
You are a machine learning explanation assistant.

Return only valid JSON.

Fields required:
prediction_label,
confidence_level,
top_reason,
second_reason,
next_step
"""


    user_prompt = f"""
Features:
{prompt_record}

Prediction:
{prediction}

Probability:
{probability}

Explain the prediction in JSON format.
"""


    output_temp0 = call_llm(
        system_prompt,
        user_prompt,
        temperature=0
    )


    output_temp07 = call_llm(
        system_prompt,
        user_prompt,
        temperature=0.7
    )


    temperature_results.append({

        "Input": i+1,

        "Output_temp_0": output_temp0,

        "Output_temp_07": output_temp07

    })


temperature_results

[{'Input': 1,
  'Output_temp_0': '{\n  "prediction_label": 1,\n  "confidence_level": 0.88,\n  "top_reason": "The model predicts a house with the given features is likely to be a single-family home.",\n  "second_reason": "The Lot Frontage and Lot Area suggest a typical suburban house.",\n  "next_step": "Further analysis of the house\'s architectural style and neighborhood would be beneficial for a more accurate prediction."\n}',
  'Output_temp_07': '{\n  "prediction_label": 1,\n  "confidence_level": 0.88,\n  "top_reason": "The top reason for this prediction is not available in the given data, but we can infer that the model is making a prediction based on the features provided.",\n  "second_reason": "The second reason for this prediction is also not available in the given data, but we can infer that the model is considering the features such as \'Order\', \'PID\', \'MS SubClass\' and \'Year Built\' to make the prediction.",\n  "next_step": "To get a more accurate explanation of the pred

## Guardrail Test
Testing PII detection before LLM calls.

In [46]:
pii_tests = [

    "My email is pallavi123@gmail.com",

    "Explain this house prediction"

]


for text in pii_tests:

    if has_pii(text):

        print("Input blocked: PII detected.")

    else:

        response = call_llm(
            "You are a helpful assistant.",
            text
        )

        print("Input allowed:")
        print(response)

Input blocked: PII detected.
Input allowed:
I'd be happy to help you understand a house prediction. However, I don't see any specific information about a house prediction. Could you please provide more context or details about the prediction you're referring to? 

Is it a prediction about a specific house, a general prediction about the housing market, or perhaps a prediction from a fortune teller or a psychic? 

Please provide more information, and I'll do my best to help you understand the prediction.
